In [ ]:
# Trabajo Sumativo N°3 - Machine Learning
# Titanic: Clustering — Algoritmos de Distancia y Densidad
# Universidad Mayor - Ingeniería Civil Industrial
# Unidad 3 - Aprendizaje No Supervisado
#
# Integrantes:
#   Manuel Herrera Liberona
#   Iván Ramirez Martinez
#   Marcelo Reyes Alcalde
#   Francisco Rojas Escalona

## Introducción

En los Trabajos N°1 y N°2 construimos un pipeline completo de análisis supervisado sobre el dataset del Titanic de Kaggle: limpieza, imputación, feature engineering, entrenamiento de modelos de clasificación (Ridge, Lasso, ElasticNet, Random Forest, XGBoost, Red Neuronal) y evaluación con métricas como F1-Score y AUC.

Ahora, en esta tercera y última etapa, abordamos el problema desde la perspectiva del **aprendizaje no supervisado (clustering)**. El objetivo es descubrir agrupaciones naturales en los datos **sin usar la variable objetivo (Survived)** y luego evaluar si esas agrupaciones coinciden con la supervivencia real de los pasajeros.

Compararemos algoritmos basados en **distancia** (K-Means, Mini-Batch K-Means, Agglomerative Clustering) y basados en **densidad** (DBSCAN, OPTICS, Mean Shift), tanto con como sin reducción de dimensionalidad mediante **PCA (Análisis de Componentes Principales)**.

---

## 0. Librerías y configuración

Importamos todas las librerías necesarias para clustering, PCA, métricas de evaluación y visualización.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
warnings.filterwarnings('ignore')

# Preprocesamiento
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# PCA
from sklearn.decomposition import PCA

# Algoritmos de Clustering
from sklearn.cluster import (KMeans, MiniBatchKMeans, AgglomerativeClustering,
                              DBSCAN, MeanShift, OPTICS, estimate_bandwidth)

# Métricas de Clustering
from sklearn.metrics import (silhouette_score, silhouette_samples,
                              calinski_harabasz_score, davies_bouldin_score,
                              adjusted_rand_score, normalized_mutual_info_score,
                              confusion_matrix, accuracy_score)

# Vecinos más cercanos (para k-distancias)
from sklearn.neighbors import NearestNeighbors

# Visualización jerárquica
from scipy.cluster.hierarchy import dendrogram, linkage

# Reproducibilidad
np.random.seed(42)

print("Librerías cargadas correctamente.")
print(f"Scikit-learn version: {__import__('sklearn').__version__}")

---

## 1. Preparación de datos

### 1a. Limpieza de datos (continuación de Unidad 1)

Replicamos el preprocesamiento establecido en el Trabajo N°1, asegurando la reproducibilidad del pipeline.

In [ ]:
# --- Cargar dataset ---
df = pd.read_csv("train.csv")
print(f"Dataset original: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"Nulos por columna:")
print(df.isnull().sum())
print(f"\nVariable objetivo (Survived): {df['Survived'].value_counts().to_dict()}")

In [ ]:
# --- Imputación de nulos (mismo criterio Unidad 1) ---
# Age: mediana (distribución asimétrica, la mediana es más robusta que la media)
df['Age'] = df['Age'].fillna(df['Age'].median())

# Embarked: moda (solo 2 nulos, imputar con el valor más frecuente es razonable)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Cabin: eliminamos (77% nulos, irrecuperable sin sesgar)
df = df.drop(columns=['Cabin'])

# --- Tratamiento de outliers con Winsorización en Fare (mismo criterio Unidad 1) ---
Q1_fare = df['Fare'].quantile(0.25)
Q3_fare = df['Fare'].quantile(0.75)
IQR_fare = Q3_fare - Q1_fare
upper_fare = Q3_fare + 1.5 * IQR_fare
lower_fare = Q1_fare - 1.5 * IQR_fare
df['Fare'] = df['Fare'].clip(lower=lower_fare, upper=upper_fare)

# --- Eliminar columnas no predictivas ---
# Name, Ticket, PassengerId: identificadores sin valor predictivo para clustering
df = df.drop(columns=["Name", "Ticket", "PassengerId"])

# --- Feature Engineering (mismo que Unidades anteriores) ---
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

print(f"Dataset limpio: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"Nulos restantes: {df.isnull().sum().sum()}")
df.head()

### 1b. One Hot Encoding

Aplicamos One Hot Encoding a las variables categóricas **Sex** y **Embarked**.

**Justificación:** Los algoritmos de clustering basados en distancia (K-Means, DBSCAN, etc.) operan con métricas euclidianas o similares, que requieren variables numéricas. One Hot Encoding transforma las categorías en columnas binarias (0/1) sin imponer un orden artificial entre categorías, a diferencia de Label Encoding que asignaría valores ordinales (0, 1, 2) a categorías nominales como los puertos de embarque.

Usamos `drop_first=True` para evitar multicolinealidad perfecta (trampa de las variables dummy), que podría distorsionar las distancias en el espacio de features.

In [ ]:
# --- Separar la variable objetivo ANTES de la codificación ---
# Guardamos 'Survived' para el análisis posterior (Punto 4c) pero NO la usamos en clustering
y_true = df['Survived'].copy()
df_clustering = df.drop(columns=['Survived'])

# --- One Hot Encoding ---
df_encoded = pd.get_dummies(df_clustering, columns=['Sex', 'Embarked'], drop_first=True)

print(f"Variables después de OHE ({df_encoded.shape[1]}):")
print(f"  {list(df_encoded.columns)}")
print(f"\nTipos de datos:")
print(df_encoded.dtypes)
df_encoded.head()

### 1c. Análisis de correlación entre variables

Antes de aplicar clustering, analizamos la correlación entre las variables para entender qué información es redundante y cuáles variables aportan información diferenciada.

Este análisis es importante porque variables altamente correlacionadas pueden sesgar los algoritmos de clustering basados en distancia, ya que el mismo "concepto" se cuenta varias veces al calcular distancias euclidianas.

In [ ]:
# --- Heatmap de correlación ---
fig, ax = plt.subplots(figsize=(12, 9))

corr_matrix = df_encoded.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            square=True, cbar_kws={'shrink': 0.8, 'label': 'Correlación de Pearson'})
ax.set_title('Matriz de Correlación entre Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Identificar pares altamente correlacionados
print("Pares de variables con correlación alta (|r| > 0.5):")
for i in range(len(corr_matrix)):
    for j in range(i+1, len(corr_matrix)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.5:
            print(f"  {corr_matrix.index[i]} <-> {corr_matrix.columns[j]}: r = {r:.3f}")

print("\nObservación: Las correlaciones altas entre SibSp/Parch/FamilySize/IsAlone son esperables")
print("ya que FamilySize = SibSp + Parch + 1 e IsAlone se deriva de FamilySize.")
print("PCA se encargará de manejar esta redundancia al comprimir la información.")

### 1d. Normalización

Aplicamos **StandardScaler** (estandarización Z-score) a todas las variables.

**Justificación:** Los algoritmos de clustering basados en distancia son sensibles a la escala de las variables. Sin normalización, variables con rangos amplios (como Fare: 0-65) dominarían el cálculo de distancias frente a variables con rangos pequeños (como IsAlone: 0-1). La estandarización Z-score (media=0, desviación estándar=1) asegura que todas las variables contribuyan equitativamente al cálculo de distancias.

Elegimos StandardScaler sobre MinMaxScaler porque:
- Es más robusto ante outliers residuales.
- PCA asume datos centrados (media=0), lo cual StandardScaler garantiza.
- Es el estándar recomendado para K-Means y algoritmos basados en distancia euclidiana.

In [ ]:
# --- Normalización con StandardScaler ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded)

# Convertir a DataFrame para mantener nombres de columnas
X_scaled_df = pd.DataFrame(X_scaled, columns=df_encoded.columns)

print("Normalización aplicada (StandardScaler).")
print(f"Media de features (debe ser ~0): {X_scaled.mean(axis=0).round(6)}")
print(f"Std de features (debe ser ~1):   {X_scaled.std(axis=0).round(4)}")
print(f"\nShape final: {X_scaled.shape}")
X_scaled_df.describe().round(3)

### 1e. Validación de tendencia al clustering (Hopkins Statistic)

Antes de aplicar cualquier algoritmo de clustering, es fundamental verificar si los datos realmente poseen una estructura agrupable o si son uniformemente distribuidos (donde el clustering no tendría sentido).

El **estadístico de Hopkins** mide la tendencia al clustering comparando las distancias al vecino más cercano en los datos reales vs. puntos generados aleatoriamente:
- **H ≈ 0.5:** Los datos son uniformes (no hay clusters naturales).
- **H > 0.7:** Alta tendencia al clustering.
- **H > 0.9:** Fuerte evidencia de clusters.

Calculamos Hopkins sobre múltiples muestras aleatorias para obtener un resultado robusto.

In [ ]:
# --- Hopkins Statistic ---
def hopkins_statistic(X, sample_size=None, n_iter=100):
    """Calcula el estadístico de Hopkins promediado sobre n_iter iteraciones."""
    n, d = X.shape
    if sample_size is None:
        sample_size = min(int(n * 0.1), 100)

    hopkins_values = []
    for _ in range(n_iter):
        # Muestra aleatoria de los datos reales
        idx = np.random.choice(n, sample_size, replace=False)
        X_sample = X[idx]

        # Puntos aleatorios uniformes en el rango de los datos
        X_random = np.random.uniform(X.min(axis=0), X.max(axis=0), size=(sample_size, d))

        # Distancia al vecino más cercano para cada conjunto
        nn_real = NearestNeighbors(n_neighbors=2)
        nn_real.fit(X)

        # Para los datos reales: distancia al 2do vecino más cercano (el 1ro es él mismo)
        dist_real, _ = nn_real.kneighbors(X_sample, n_neighbors=2)
        u = dist_real[:, 1].sum()

        # Para los puntos aleatorios: distancia al vecino más cercano
        dist_random, _ = nn_real.kneighbors(X_random, n_neighbors=1)
        w = dist_random[:, 0].sum()

        hopkins_values.append(w / (u + w))

    return np.mean(hopkins_values), np.std(hopkins_values)

# Calcular Hopkins para ambos datasets
h_sin_pca, h_std_sin = hopkins_statistic(X_scaled)
print("=" * 60)
print("VALIDACIÓN DE TENDENCIA AL CLUSTERING (Hopkins Statistic)")
print("=" * 60)
print(f"  Sin PCA: H = {h_sin_pca:.4f} ± {h_std_sin:.4f}")

if h_sin_pca > 0.9:
    interpretacion = "Fuerte evidencia de estructura agrupable"
elif h_sin_pca > 0.7:
    interpretacion = "Alta tendencia al clustering"
elif h_sin_pca > 0.5:
    interpretacion = "Tendencia moderada al clustering"
else:
    interpretacion = "No hay evidencia clara de clusters"

print(f"  Interpretación: {interpretacion}")
print(f"\nConclusión: Con H = {h_sin_pca:.4f}, se confirma que los datos del Titanic")
print("poseen estructura agrupable y tiene sentido aplicar algoritmos de clustering.")

---

## 2. Experimento con y sin PCA

### ¿Qué es PCA?

PCA (Análisis de Componentes Principales) es una técnica de reducción de dimensionalidad que transforma las variables originales en nuevas variables (componentes principales) que son combinaciones lineales de las originales, ordenadas por la cantidad de varianza que explican.

**¿Por qué usar PCA en clustering?**
- **Reduce el ruido:** Elimina dimensiones que aportan poca información, mejorando la señal.
- **Mitiga la maldición de la dimensionalidad:** En espacios de alta dimensión, las distancias se vuelven menos significativas.
- **Elimina redundancia:** Como vimos en la correlación, variables como SibSp, Parch, FamilySize e IsAlone comparten información. PCA comprime esa redundancia.
- **Facilita la visualización:** Permite representar los clusters en 2D/3D.

### 2a. Análisis de varianza explicada

In [ ]:
# --- Análisis de varianza explicada ---
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

# Varianza explicada por cada componente
var_explicada = pca_full.explained_variance_ratio_
var_acumulada = np.cumsum(var_explicada)

print("Varianza explicada por componente:")
for i, (ve, va) in enumerate(zip(var_explicada, var_acumulada)):
    print(f"  PC{i+1}: {ve:.4f} ({ve*100:.2f}%)  |  Acumulada: {va:.4f} ({va*100:.2f}%)")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, len(var_explicada)+1), var_explicada, color='#3498db', alpha=0.8)
axes[0].plot(range(1, len(var_explicada)+1), var_explicada, 'ro-', markersize=6)
axes[0].set_xlabel('Componente Principal', fontsize=12)
axes[0].set_ylabel('Varianza Explicada (%)', fontsize=12)
axes[0].set_title('Scree Plot - Varianza por Componente', fontsize=13, fontweight='bold')
axes[0].set_xticks(range(1, len(var_explicada)+1))
axes[0].grid(True, alpha=0.3)

# Varianza acumulada
axes[1].plot(range(1, len(var_acumulada)+1), var_acumulada, 'go-', markersize=8, linewidth=2)
axes[1].axhline(y=0.70, color='blue', linestyle='--', alpha=0.5, label='70% varianza')
axes[1].axhline(y=0.80, color='red', linestyle='--', alpha=0.7, label='80% varianza')
axes[1].axhline(y=0.90, color='orange', linestyle='--', alpha=0.7, label='90% varianza')
axes[1].axhline(y=0.95, color='green', linestyle='--', alpha=0.7, label='95% varianza')
axes[1].fill_between(range(1, len(var_acumulada)+1), var_acumulada, alpha=0.1, color='green')
axes[1].set_xlabel('Número de Componentes', fontsize=12)
axes[1].set_ylabel('Varianza Acumulada', fontsize=12)
axes[1].set_title('Varianza Acumulada por Componentes', fontsize=13, fontweight='bold')
axes[1].set_xticks(range(1, len(var_acumulada)+1))
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2b. Heatmap de Loadings: ¿Qué mide cada componente?

Los **loadings** (cargas) indican cuánto contribuye cada variable original a cada componente principal. Esto permite interpretar semánticamente qué "concepto" captura cada PC.

In [ ]:
# --- Heatmap de loadings ---
loadings_full = pd.DataFrame(
    pca_full.components_.T,
    index=df_encoded.columns,
    columns=[f'PC{i+1}' for i in range(len(df_encoded.columns))]
)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(loadings_full, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax, vmin=-0.7, vmax=0.7,
            cbar_kws={'shrink': 0.8, 'label': 'Loading'})
ax.set_title('Heatmap de Loadings: Contribución de Variables a Componentes Principales',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Variable Original')
ax.set_xlabel('Componente Principal')
plt.tight_layout()
plt.show()

# --- Interpretación semántica ---
print("=" * 70)
print("INTERPRETACIÓN SEMÁNTICA DE LOS COMPONENTES PRINCIPALES")
print("=" * 70)

for i in range(min(4, len(df_encoded.columns))):
    pc = loadings_full.iloc[:, i]
    top_pos = pc.nlargest(3)
    top_neg = pc.nsmallest(3)
    print(f"\nPC{i+1} ({var_explicada[i]*100:.1f}% de varianza):")
    print(f"  Cargas positivas más fuertes:")
    for feat, val in top_pos.items():
        print(f"    {feat:20s}: {val:+.3f}")
    print(f"  Cargas negativas más fuertes:")
    for feat, val in top_neg.items():
        print(f"    {feat:20s}: {val:+.3f}")

### 2c. Comparación de umbrales de varianza

Evaluamos cómo cambia la calidad del clustering (K-Means con K=2) al variar el número de componentes seleccionados. Esto nos permite elegir el umbral óptimo de varianza.

In [ ]:
# --- Comparación de umbrales ---
umbrales = [0.70, 0.80, 0.90, 0.95, 1.00]
resultados_umbrales = []

for umbral in umbrales:
    if umbral >= 1.0:
        n_comp = X_scaled.shape[1]
        X_temp = X_scaled.copy()
        var_real = 1.0
    else:
        n_comp = np.argmax(var_acumulada >= umbral) + 1
        pca_temp = PCA(n_components=n_comp, random_state=42)
        X_temp = pca_temp.fit_transform(X_scaled)
        var_real = var_acumulada[n_comp - 1]

    km = KMeans(n_clusters=2, random_state=42, n_init=10)
    labels_temp = km.fit_predict(X_temp)
    sil = silhouette_score(X_temp, labels_temp)
    ari = adjusted_rand_score(y_true, labels_temp)
    ch = calinski_harabasz_score(X_temp, labels_temp)
    db = davies_bouldin_score(X_temp, labels_temp)

    resultados_umbrales.append({
        'Umbral': f'{umbral*100:.0f}%',
        'N_Componentes': n_comp,
        'Var_Real': f'{var_real*100:.1f}%',
        'Silhouette': sil,
        'Calinski-Harabasz': ch,
        'Davies-Bouldin': db,
        'ARI': ari
    })

df_umbrales = pd.DataFrame(resultados_umbrales)
print("Impacto del número de componentes PCA en la calidad del clustering (K-Means, K=2):")
print(df_umbrales.to_string(index=False))

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
x_labels = df_umbrales['Umbral'].values

axes[0].bar(x_labels, df_umbrales['Silhouette'], color='#3498db', alpha=0.8)
axes[0].set_title('Silhouette Score', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Umbral de Varianza')
axes[0].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(df_umbrales['Silhouette']):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=9)

axes[1].bar(x_labels, df_umbrales['Davies-Bouldin'], color='#e74c3c', alpha=0.8)
axes[1].set_title('Davies-Bouldin (menor = mejor)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Umbral de Varianza')
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(df_umbrales['Davies-Bouldin']):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)

axes[2].bar(x_labels, df_umbrales['ARI'], color='#2ecc71', alpha=0.8)
axes[2].set_title('ARI vs Survived', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Umbral de Varianza')
axes[2].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(df_umbrales['ARI']):
    axes[2].text(i, v + 0.005 if v >= 0 else v - 0.02, f'{v:.3f}', ha='center', fontsize=9)

plt.suptitle('Efecto del Umbral de Varianza PCA en la Calidad del Clustering',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2d. Selección del número de componentes y preparación de datasets

In [ ]:
# --- Selección: 80% de varianza ---
umbral_varianza = 0.80
n_componentes = np.argmax(var_acumulada >= umbral_varianza) + 1

print(f"Umbral de varianza seleccionado: {umbral_varianza*100:.0f}%")
print(f"Número de componentes: {n_componentes}")
print(f"Varianza explicada: {var_acumulada[n_componentes-1]*100:.2f}%")
print(f"Reducción: {X_scaled.shape[1]} → {n_componentes} variables ({(1 - n_componentes/X_scaled.shape[1])*100:.1f}% de reducción)")

# --- Aplicar PCA ---
pca = PCA(n_components=n_componentes, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Loadings del PCA seleccionado
loadings = pd.DataFrame(
    pca.components_.T,
    index=df_encoded.columns,
    columns=[f'PC{i+1}' for i in range(n_componentes)]
)
print(f"\nLoadings (contribución de cada variable a los componentes seleccionados):")
print(loadings.round(3).to_string())

print(f"\nShape SIN PCA: {X_scaled.shape}")
print(f"Shape CON PCA: {X_pca.shape}")

In [ ]:
# --- Visualización en 2D (primeras 2 componentes) ---
pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Coloreado por Survived (referencia visual)
scatter1 = axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=y_true, cmap='coolwarm',
                            alpha=0.5, s=20, edgecolors='none')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11)
axes[0].set_title('Datos en 2D (PCA) - Coloreado por Survived', fontsize=13, fontweight='bold')
axes[0].legend(*scatter1.legend_elements(), title='Survived', loc='upper right')
axes[0].grid(True, alpha=0.3)

# Biplot
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.15, s=10, color='gray')
for i, col in enumerate(df_encoded.columns):
    axes[1].arrow(0, 0, pca_2d.components_[0, i]*5, pca_2d.components_[1, i]*5,
                  head_width=0.15, head_length=0.1, fc='red', ec='red', alpha=0.7)
    axes[1].text(pca_2d.components_[0, i]*5.5, pca_2d.components_[1, i]*5.5,
                 col, fontsize=8, ha='center', color='darkred')
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)', fontsize=11)
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11)
axes[1].set_title('Biplot - Dirección de Variables Originales', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 3. Challenge entre algoritmos de clustering (Benchmarking)

Realizamos un benchmarking entre **6 algoritmos de clustering**, clasificados en dos familias:

**Algoritmos basados en distancia:**
1. **K-Means:** Particiona los datos en K clusters minimizando la suma de distancias al centroide. Asume clusters esféricos de tamaño similar.
2. **Mini-Batch K-Means:** Variante escalable de K-Means que usa subconjuntos (mini-batches) para actualizar los centroides, más rápido pero ligeramente menos preciso.
3. **Agglomerative Clustering (Ward):** Enfoque jerárquico bottom-up que fusiona los clusters más similares iterativamente. El criterio Ward minimiza la varianza intra-cluster.

**Algoritmos basados en densidad:**
4. **DBSCAN:** Agrupa puntos en regiones de alta densidad separadas por regiones de baja densidad. Puede detectar clusters de forma arbitraria y marcar outliers como ruido (-1).
5. **OPTICS:** Generalización de DBSCAN que ordena los puntos por densidad de alcanzabilidad. Puede detectar clusters de diferentes densidades.
6. **Mean Shift:** Busca modas (máximos) de la función de densidad. Determina automáticamente el número de clusters.

Todos los algoritmos se aplican sobre las **dos versiones** del dataset (Sin PCA y Con PCA).

### 3a. Determinación del número óptimo de clusters (K)

Determinamos el K óptimo usando el **método del codo** y el **coeficiente de silueta**.

In [ ]:
# --- Método del Codo y Silueta para determinar K óptimo ---
K_range = range(2, 11)
inertias = []
silhouettes_k = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes_k.append(silhouette_score(X_scaled, labels_k))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Método del codo
axes[0].plot(K_range, inertias, 'bo-', markersize=8, linewidth=2)
axes[0].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inercia (SSE)', fontsize=12)
axes[0].set_title('Método del Codo', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Coeficiente de silueta
axes[1].plot(K_range, silhouettes_k, 'rs-', markersize=8, linewidth=2)
axes[1].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[1].set_ylabel('Coeficiente de Silueta', fontsize=12)
axes[1].set_title('Silueta Promedio por K', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

k_optimo = K_range[np.argmax(silhouettes_k)]
print(f"K con mejor silueta: {k_optimo} (Silueta = {max(silhouettes_k):.4f})")
print(f"\nUsaremos K=2 como referencia principal para comparar con la variable binaria Survived,")
print(f"y K={k_optimo} como alternativa basada en la silueta óptima.")

### 3a-bis. Ejecución con K óptimo vs K=2

Además de K=2 (que coincide con la variable binaria Survived), ejecutamos los algoritmos basados en distancia con el K óptimo identificado por la silueta. Esto permite comparar si la estructura natural de los datos sugiere 2 grupos o más.

In [ ]:
# --- Comparar K=2 vs K óptimo con los algoritmos de distancia ---
if k_optimo != 2:
    print(f"Comparando K=2 vs K={k_optimo} para algoritmos basados en distancia")
    print("=" * 80)

    resultados_k = {}
    for k_val in [2, k_optimo]:
        for nombre_algo, AlgoClass, kwargs in [
            ('K-Means', KMeans, {'n_init': 10, 'random_state': 42}),
            ('Mini-Batch KM', MiniBatchKMeans, {'batch_size': 100, 'n_init': 10, 'random_state': 42}),
            ('Agglomerative', AgglomerativeClustering, {'linkage': 'ward'}),
        ]:
            modelo = AlgoClass(n_clusters=k_val, **kwargs)
            labels_k = modelo.fit_predict(X_scaled)
            sil = silhouette_score(X_scaled, labels_k)
            ch = calinski_harabasz_score(X_scaled, labels_k)
            db = davies_bouldin_score(X_scaled, labels_k)
            ari = adjusted_rand_score(y_true, labels_k)
            key = f"{nombre_algo} (K={k_val})"
            resultados_k[key] = {'Silhouette': sil, 'Calinski-Harabasz': ch,
                                  'Davies-Bouldin': db, 'ARI': ari}

    df_k_comp = pd.DataFrame(resultados_k).T
    print(df_k_comp.round(4).to_string())

    # Visualización
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for idx, metrica in enumerate(['Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin', 'ARI']):
        vals = df_k_comp[metrica].values
        colors = ['#3498db' if 'K=2' in k else '#e74c3c' for k in df_k_comp.index]
        axes[idx].barh(range(len(df_k_comp)), vals, color=colors, alpha=0.8)
        axes[idx].set_yticks(range(len(df_k_comp)))
        axes[idx].set_yticklabels(df_k_comp.index, fontsize=8)
        axes[idx].set_title(metrica, fontsize=12, fontweight='bold')
        axes[idx].grid(True, alpha=0.3, axis='x')
        for i, v in enumerate(vals):
            axes[idx].text(v, i, f' {v:.3f}', va='center', fontsize=8)
    plt.suptitle(f'Comparación K=2 vs K={k_optimo}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"\nAnálisis: Si K={k_optimo} presenta mejor Silhouette pero peor ARI,")
    print("significa que los datos tienen una estructura natural diferente a la supervivencia,")
    print("lo cual es esperable: el clustering descubre patrones sociodemográficos, no causales.")
else:
    print("K óptimo = 2, coincide con la variable binaria Survived.")
    print("No se requiere comparación adicional.")

### 3b. Dendrograma (Clustering Jerárquico)

In [ ]:
# --- Dendrograma ---
fig, ax = plt.subplots(figsize=(14, 6))

np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), size=min(200, len(X_scaled)), replace=False)
X_sample = X_scaled[sample_idx]

linked = linkage(X_sample, method='ward')
dendrogram(linked, truncate_mode='level', p=5, ax=ax,
           color_threshold=10, above_threshold_color='gray')
ax.set_xlabel('Muestras (truncado)', fontsize=12)
ax.set_ylabel('Distancia (Ward)', fontsize=12)
ax.set_title('Dendrograma - Clustering Jerárquico (Ward)', fontsize=14, fontweight='bold')
ax.axhline(y=10, color='red', linestyle='--', alpha=0.7, label='Corte sugerido')
ax.legend()
plt.tight_layout()
plt.show()

print("El dendrograma confirma visualmente la existencia de 2-3 agrupaciones principales en los datos.")

### 3c. Selección de hiperparámetros para algoritmos de densidad

#### Gráfico de K-distancias para DBSCAN

El parámetro **eps** (radio de vecindad) de DBSCAN es crítico. El método de las **k-distancias** consiste en calcular la distancia al k-ésimo vecino más cercano para cada punto, ordenarlas, y buscar el "codo" en la curva: el punto donde la distancia crece abruptamente indica la separación entre puntos densos (cluster) y puntos aislados (ruido).

In [ ]:
# --- Gráfico de k-distancias para selección de eps ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (X_data, nombre) in enumerate([(X_scaled, 'Sin PCA'), (X_pca, 'Con PCA')]):
    nn = NearestNeighbors(n_neighbors=5)
    nn.fit(X_data)
    distances, _ = nn.kneighbors(X_data)
    k_distances = np.sort(distances[:, -1])

    axes[idx].plot(range(len(k_distances)), k_distances, color='#3498db', linewidth=1.5)
    axes[idx].set_xlabel('Puntos (ordenados por distancia)', fontsize=11)
    axes[idx].set_ylabel('Distancia al 5° vecino más cercano', fontsize=11)
    axes[idx].set_title(f'K-Distancias ({nombre})', fontsize=13, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)

    # Marcar percentiles candidatos para eps
    for pct, color, ls in [(85, 'orange', '--'), (90, 'red', '--'), (95, 'darkred', ':')]:
        eps_val = np.percentile(k_distances, pct)
        axes[idx].axhline(y=eps_val, color=color, linestyle=ls, alpha=0.7,
                          label=f'P{pct} = {eps_val:.2f}')
    axes[idx].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("Interpretación: El codo en la curva indica el eps óptimo.")
print("Valores por debajo del codo capturan la densidad intra-cluster;")
print("valores por encima incluirían ruido como parte de los clusters.")

#### Tuning de DBSCAN: Exploración de eps y min_samples

In [ ]:
# --- Tuning de DBSCAN ---
eps_candidates = []
for X_data in [X_scaled, X_pca]:
    nn = NearestNeighbors(n_neighbors=5)
    nn.fit(X_data)
    distances, _ = nn.kneighbors(X_data)
    eps_candidates.append([
        np.percentile(distances[:, -1], p) for p in [80, 85, 90, 95]
    ])

min_samples_options = [3, 5, 10]

print("=" * 80)
print("TUNING DE DBSCAN: Exploración de eps y min_samples")
print("=" * 80)

best_dbscan = {'sin_pca': None, 'con_pca': None}

for data_idx, (X_data, nombre) in enumerate([(X_scaled, 'Sin PCA'), (X_pca, 'Con PCA')]):
    print(f"\n--- {nombre} ---")
    best_sil = -1

    for eps_val in eps_candidates[data_idx]:
        for ms in min_samples_options:
            db = DBSCAN(eps=eps_val, min_samples=ms)
            labels_db = db.fit_predict(X_data)
            n_clusters = len(set(labels_db) - {-1})
            n_noise = np.sum(labels_db == -1)
            pct_noise = n_noise / len(labels_db) * 100

            mask = labels_db != -1
            if n_clusters >= 2 and mask.sum() > n_clusters:
                sil = silhouette_score(X_data[mask], labels_db[mask])
                ari = adjusted_rand_score(y_true, labels_db)
                print(f"  eps={eps_val:.2f}, min_samples={ms:2d} → "
                      f"Clusters: {n_clusters}, Ruido: {n_noise} ({pct_noise:.1f}%), "
                      f"Silhouette: {sil:.4f}, ARI: {ari:.4f}")
                if sil > best_sil:
                    best_sil = sil
                    key = 'sin_pca' if data_idx == 0 else 'con_pca'
                    best_dbscan[key] = {'eps': eps_val, 'min_samples': ms, 'sil': sil}
            else:
                print(f"  eps={eps_val:.2f}, min_samples={ms:2d} → "
                      f"Clusters: {n_clusters}, Ruido: {n_noise} ({pct_noise:.1f}%) [insuficiente]")

for key in ['sin_pca', 'con_pca']:
    if best_dbscan[key]:
        print(f"\nMejor DBSCAN ({key}): eps={best_dbscan[key]['eps']:.2f}, "
              f"min_samples={best_dbscan[key]['min_samples']}, Silhouette={best_dbscan[key]['sil']:.4f}")

### 3d. Aplicación de los 6 algoritmos (con y sin PCA) y tiempos de ejecución

In [ ]:
# --- Función para aplicar todos los algoritmos con timing ---
def aplicar_clustering(X_data, nombre_datos, k=2, dbscan_params=None):
    resultados = {}
    tiempos = {}

    # === DISTANCIA ===
    # 1. K-Means
    t0 = time.time()
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    resultados['K-Means'] = {'labels': km.fit_predict(X_data), 'tipo': 'Distancia', 'modelo': km}
    tiempos['K-Means'] = time.time() - t0

    # 2. Mini-Batch K-Means
    t0 = time.time()
    mbkm = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=100, n_init=10)
    resultados['Mini-Batch KM'] = {'labels': mbkm.fit_predict(X_data), 'tipo': 'Distancia', 'modelo': mbkm}
    tiempos['Mini-Batch KM'] = time.time() - t0

    # 3. Agglomerative Clustering
    t0 = time.time()
    agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
    resultados['Agglomerative'] = {'labels': agg.fit_predict(X_data), 'tipo': 'Distancia', 'modelo': agg}
    tiempos['Agglomerative'] = time.time() - t0

    # === DENSIDAD ===
    # 4. DBSCAN (con hiperparámetros tuneados)
    if dbscan_params:
        eps_val = dbscan_params['eps']
        ms_val = dbscan_params['min_samples']
    else:
        nn = NearestNeighbors(n_neighbors=5)
        nn.fit(X_data)
        distances, _ = nn.kneighbors(X_data)
        eps_val = np.percentile(distances[:, -1], 90)
        ms_val = 5

    t0 = time.time()
    dbscan = DBSCAN(eps=eps_val, min_samples=ms_val)
    resultados['DBSCAN'] = {'labels': dbscan.fit_predict(X_data), 'tipo': 'Densidad', 'modelo': dbscan}
    tiempos['DBSCAN'] = time.time() - t0

    # 5. OPTICS
    t0 = time.time()
    optics = OPTICS(min_samples=5, xi=0.05, min_cluster_size=0.05)
    resultados['OPTICS'] = {'labels': optics.fit_predict(X_data), 'tipo': 'Densidad', 'modelo': optics}
    tiempos['OPTICS'] = time.time() - t0

    # 6. Mean Shift
    t0 = time.time()
    bandwidth = estimate_bandwidth(X_data, quantile=0.3, random_state=42)
    if bandwidth == 0:
        bandwidth = 1.0
    ms = MeanShift(bandwidth=bandwidth)
    resultados['Mean Shift'] = {'labels': ms.fit_predict(X_data), 'tipo': 'Densidad', 'modelo': ms}
    tiempos['Mean Shift'] = time.time() - t0

    print(f"\n{'='*75}")
    print(f"Resultados de Clustering - {nombre_datos}")
    print(f"{'='*75}")
    print(f"  {'Algoritmo':20s} | {'Tipo':10s} | {'Clusters':>8s} | {'Ruido':>6s} | {'Tiempo':>10s}")
    print(f"  {'-'*20}-+-{'-'*10}-+-{'-'*8}-+-{'-'*6}-+-{'-'*10}")
    for nombre, res in resultados.items():
        labels = res['labels']
        n_clusters = len(set(labels) - {-1})
        n_noise = np.sum(labels == -1)
        t = tiempos[nombre]
        print(f"  {nombre:20s} | {res['tipo']:10s} | {n_clusters:8d} | {n_noise:6d} | {t*1000:8.1f} ms")

    return resultados, tiempos

# --- Aplicar SIN PCA ---
resultados_sin_pca, tiempos_sin_pca = aplicar_clustering(
    X_scaled, "SIN PCA", k=2, dbscan_params=best_dbscan['sin_pca'])

# --- Aplicar CON PCA ---
resultados_con_pca, tiempos_con_pca = aplicar_clustering(
    X_pca, "CON PCA", k=2, dbscan_params=best_dbscan['con_pca'])

In [ ]:
# --- Comparación de tiempos de ejecución ---
nombres_algo = list(tiempos_sin_pca.keys())

fig, ax = plt.subplots(figsize=(12, 5))
x_pos = np.arange(len(nombres_algo))
width = 0.35

t_sp = [tiempos_sin_pca[n]*1000 for n in nombres_algo]
t_cp = [tiempos_con_pca[n]*1000 for n in nombres_algo]

bars1 = ax.bar(x_pos - width/2, t_sp, width, label='Sin PCA', color='#3498db', alpha=0.8)
bars2 = ax.bar(x_pos + width/2, t_cp, width, label='Con PCA', color='#e74c3c', alpha=0.8)

ax.set_xlabel('Algoritmo', fontsize=12)
ax.set_ylabel('Tiempo de ejecución (ms)', fontsize=12)
ax.set_title('Tiempos de Ejecución por Algoritmo', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(nombres_algo, rotation=30, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{bar.get_height():.0f}',
            ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{bar.get_height():.0f}',
            ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print("Observaciones sobre tiempos:")
print("  - Mini-Batch K-Means es el más rápido (diseñado para escalabilidad).")
print("  - OPTICS y Mean Shift suelen ser los más lentos (cálculos de densidad más complejos).")
print("  - PCA reduce dimensionalidad y generalmente acelera el cómputo.")

In [ ]:
# --- Visualización de clusters en 2D ---
fig, axes = plt.subplots(2, 6, figsize=(24, 10))

for idx, nombre in enumerate(nombres_algo):
    # Sin PCA
    ax = axes[0, idx]
    labels = resultados_sin_pca[nombre]['labels']
    scatter = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap='Set1',
                         alpha=0.5, s=15, edgecolors='none')
    tipo = resultados_sin_pca[nombre]['tipo']
    n_cl = len(set(labels) - {-1})
    ax.set_title(f'{nombre}\n({tipo}, K={n_cl})', fontsize=10, fontweight='bold')
    if idx == 0:
        ax.set_ylabel('SIN PCA\nPC2', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.2)

    # Con PCA
    ax2 = axes[1, idx]
    labels_pca = resultados_con_pca[nombre]['labels']
    scatter2 = ax2.scatter(X_2d[:, 0], X_2d[:, 1], c=labels_pca, cmap='Set1',
                           alpha=0.5, s=15, edgecolors='none')
    n_cl_pca = len(set(labels_pca) - {-1})
    ax2.set_title(f'{nombre}\n({tipo}, K={n_cl_pca})', fontsize=10, fontweight='bold')
    if idx == 0:
        ax2.set_ylabel('CON PCA\nPC2', fontsize=11, fontweight='bold')
    ax2.set_xlabel('PC1', fontsize=10)
    ax2.grid(True, alpha=0.2)

plt.suptitle('Comparación de Algoritmos de Clustering: Sin PCA vs Con PCA',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 3e. Reachability Plot - OPTICS

El **Reachability Plot** es la visualización principal de OPTICS. Muestra la distancia de alcanzabilidad de cada punto en el orden procesado por el algoritmo. Los **valles** representan clusters (puntos con baja distancia de alcanzabilidad, es decir, regiones densas) y los **picos** representan fronteras entre clusters o puntos de ruido.

In [ ]:
# --- Reachability Plot para OPTICS ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for idx, (X_data, nombre, resultados) in enumerate([
    (X_scaled, 'Sin PCA', resultados_sin_pca),
    (X_pca, 'Con PCA', resultados_con_pca)
]):
    ax = axes[idx]
    modelo_optics = resultados['OPTICS']['modelo']
    reachability = modelo_optics.reachability_[modelo_optics.ordering_]
    labels_ordered = modelo_optics.labels_[modelo_optics.ordering_]

    # Colorear por cluster asignado
    colors_reach = plt.cm.Set1(labels_ordered / max(labels_ordered.max(), 1))
    # Ruido en gris
    colors_reach[labels_ordered == -1] = [0.7, 0.7, 0.7, 1.0]

    ax.bar(range(len(reachability)), reachability, color=colors_reach, width=1.0, edgecolor='none')
    ax.set_xlabel('Puntos (ordenados por OPTICS)', fontsize=11)
    ax.set_ylabel('Distancia de Alcanzabilidad', fontsize=11)
    ax.set_title(f'Reachability Plot - OPTICS ({nombre})', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    n_clusters_o = len(set(modelo_optics.labels_) - {-1})
    n_noise_o = np.sum(modelo_optics.labels_ == -1)
    ax.text(0.98, 0.95, f'Clusters: {n_clusters_o}\nRuido: {n_noise_o}',
            transform=ax.transAxes, ha='right', va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("Interpretación del Reachability Plot:")
print("  - Los valles (zonas bajas) representan clusters densos.")
print("  - Los picos altos representan fronteras entre clusters o puntos aislados.")
print("  - La profundidad del valle indica la densidad del cluster.")
print("  - Valles de diferente profundidad indican clusters de diferente densidad,")
print("    que es justamente la fortaleza de OPTICS frente a DBSCAN.")

---

## 4. Evaluación con métricas de clustering

### Métricas utilizadas

**Métricas internas (sin etiquetas):**
- **Silhouette Score** [-1, 1]: Mide qué tan similar es cada punto a su cluster vs. el cluster más cercano. >0.5 bueno, >0.25 aceptable.
- **Calinski-Harabasz Index** [0, ∞): Ratio dispersión inter/intra-cluster. Mayor = mejor.
- **Davies-Bouldin Index** [0, ∞): Similitud promedio entre clusters. Menor = mejor.

**Métricas externas (vs Survived):**
- **Adjusted Rand Index (ARI)** [-1, 1]: Concordancia ajustada por azar. 1 = perfecto, 0 = aleatorio.
- **Normalized Mutual Information (NMI)** [0, 1]: Información compartida normalizada. 1 = perfecta.

In [ ]:
# --- Calcular métricas para todos los modelos ---
def calcular_metricas_clustering(X_data, labels, y_real, nombre_modelo):
    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    n_noise = np.sum(~mask)

    metricas = {
        'Modelo': nombre_modelo,
        'N_Clusters': n_clusters,
        'N_Ruido': n_noise,
    }

    if n_clusters >= 2 and mask.sum() > n_clusters:
        metricas['Silhouette'] = silhouette_score(X_data[mask], labels[mask])
        metricas['Calinski-Harabasz'] = calinski_harabasz_score(X_data[mask], labels[mask])
        metricas['Davies-Bouldin'] = davies_bouldin_score(X_data[mask], labels[mask])
    else:
        metricas['Silhouette'] = np.nan
        metricas['Calinski-Harabasz'] = np.nan
        metricas['Davies-Bouldin'] = np.nan

    if n_clusters >= 2:
        metricas['ARI'] = adjusted_rand_score(y_real, labels)
        metricas['NMI'] = normalized_mutual_info_score(y_real, labels)
    else:
        metricas['ARI'] = np.nan
        metricas['NMI'] = np.nan

    return metricas

metricas_sin_pca = []
metricas_con_pca = []

for nombre in nombres_algo:
    labels_sp = resultados_sin_pca[nombre]['labels']
    m_sp = calcular_metricas_clustering(X_scaled, labels_sp, y_true, nombre)
    m_sp['Datos'] = 'Sin PCA'
    metricas_sin_pca.append(m_sp)

    labels_cp = resultados_con_pca[nombre]['labels']
    m_cp = calcular_metricas_clustering(X_pca, labels_cp, y_true, nombre)
    m_cp['Datos'] = 'Con PCA'
    metricas_con_pca.append(m_cp)

df_metricas_sp = pd.DataFrame(metricas_sin_pca).set_index('Modelo')
print("=" * 90)
print("MÉTRICAS DE CLUSTERING - SIN PCA")
print("=" * 90)
print(df_metricas_sp[['N_Clusters', 'N_Ruido', 'Silhouette', 'Calinski-Harabasz',
                       'Davies-Bouldin', 'ARI', 'NMI']].round(4).to_string())

print()

df_metricas_cp = pd.DataFrame(metricas_con_pca).set_index('Modelo')
print("=" * 90)
print("MÉTRICAS DE CLUSTERING - CON PCA")
print("=" * 90)
print(df_metricas_cp[['N_Clusters', 'N_Ruido', 'Silhouette', 'Calinski-Harabasz',
                       'Davies-Bouldin', 'ARI', 'NMI']].round(4).to_string())

In [ ]:
# --- Visualización comparativa de métricas ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metricas_plot = ['Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin', 'ARI', 'NMI']

for idx, metrica in enumerate(metricas_plot):
    ax = axes[idx // 3, idx % 3]

    vals_sp = df_metricas_sp[metrica].values
    vals_cp = df_metricas_cp[metrica].values
    x_pos = np.arange(len(nombres_algo))
    width = 0.35

    bars1 = ax.bar(x_pos - width/2, vals_sp, width, label='Sin PCA', color='#3498db', alpha=0.8)
    bars2 = ax.bar(x_pos + width/2, vals_cp, width, label='Con PCA', color='#e74c3c', alpha=0.8)

    ax.set_title(metrica, fontsize=13, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(nombres_algo, rotation=45, ha='right', fontsize=9)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

    for bar in bars1:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width()/2, h, f'{h:.3f}',
                    ha='center', va='bottom', fontsize=7)
    for bar in bars2:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width()/2, h, f'{h:.3f}',
                    ha='center', va='bottom', fontsize=7)

axes[1, 2].axis('off')

plt.suptitle('Comparación de Métricas: Sin PCA vs Con PCA', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 4b. Selección del mejor modelo

In [ ]:
# --- Selección del mejor modelo ---
df_todos = pd.concat([df_metricas_sp, df_metricas_cp])
df_validos = df_todos.dropna(subset=['Silhouette', 'ARI']).copy()

if len(df_validos) > 0:
    for col in ['Silhouette', 'ARI', 'NMI']:
        col_min = df_validos[col].min()
        col_max = df_validos[col].max()
        if col_max > col_min:
            df_validos[f'{col}_norm'] = (df_validos[col] - col_min) / (col_max - col_min)
        else:
            df_validos[f'{col}_norm'] = 0.5

    db_min = df_validos['Davies-Bouldin'].min()
    db_max = df_validos['Davies-Bouldin'].max()
    if db_max > db_min:
        df_validos['DB_norm'] = 1 - (df_validos['Davies-Bouldin'] - db_min) / (db_max - db_min)
    else:
        df_validos['DB_norm'] = 0.5

    df_validos['Score'] = (
        0.25 * df_validos['Silhouette_norm'] +
        0.25 * df_validos['DB_norm'] +
        0.25 * df_validos['ARI_norm'] +
        0.25 * df_validos['NMI_norm']
    )

    df_ranking = df_validos[['Datos', 'N_Clusters', 'Silhouette', 'Davies-Bouldin',
                              'ARI', 'NMI', 'Score']].sort_values('Score', ascending=False)

    print("=" * 90)
    print("RANKING DE MODELOS (Score Compuesto)")
    print("=" * 90)
    print(df_ranking.round(4).to_string())

    mejor = df_ranking.index[0]
    mejor_datos = df_ranking.iloc[0]['Datos']
    print(f"\n{'='*60}")
    print(f"MEJOR MODELO: {mejor} ({mejor_datos})")
    print(f"  Score Compuesto: {df_ranking.iloc[0]['Score']:.4f}")
    print(f"  Silhouette:     {df_ranking.iloc[0]['Silhouette']:.4f}")
    print(f"  Davies-Bouldin: {df_ranking.iloc[0]['Davies-Bouldin']:.4f}")
    print(f"  ARI:            {df_ranking.iloc[0]['ARI']:.4f}")
    print(f"  NMI:            {df_ranking.iloc[0]['NMI']:.4f}")
    print(f"{'='*60}")

### 4c. Silhouette Plot por muestra

El silhouette plot muestra el coeficiente de silueta **de cada punto individual**, agrupado por cluster. Esto revela la calidad interna de cada cluster: puntos con silueta negativa están mal asignados.

In [ ]:
# --- Silhouette plot para el mejor modelo y K-Means ---
if mejor_datos == 'Sin PCA':
    labels_mejor = resultados_sin_pca[mejor]['labels']
    X_mejor = X_scaled
else:
    labels_mejor = resultados_con_pca[mejor]['labels']
    X_mejor = X_pca

# También K-Means Sin PCA como referencia
labels_kmeans_sp = resultados_sin_pca['K-Means']['labels']
labels_kmeans_cp = resultados_con_pca['K-Means']['labels']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

modelos_sil = [
    ('K-Means (Sin PCA)', labels_kmeans_sp, X_scaled),
    (f'{mejor} ({mejor_datos})', labels_mejor, X_mejor)
]

for ax_idx, (nombre_m, labels_m, X_m) in enumerate(modelos_sil):
    ax = axes[ax_idx]

    # Filtrar ruido
    mask = labels_m != -1
    labels_clean = labels_m[mask]
    X_clean = X_m[mask]

    n_clusters_m = len(set(labels_clean))
    if n_clusters_m < 2:
        ax.set_title(f'{nombre_m}\n(Solo 1 cluster, no se puede graficar)')
        continue

    sil_samples = silhouette_samples(X_clean, labels_clean)
    sil_avg = silhouette_score(X_clean, labels_clean)

    y_lower = 10
    colors = plt.cm.Set1(np.linspace(0, 1, n_clusters_m))

    for i, cluster_id in enumerate(sorted(set(labels_clean))):
        cluster_sil = sil_samples[labels_clean == cluster_id]
        cluster_sil.sort()
        size = len(cluster_sil)
        y_upper = y_lower + size

        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                         facecolor=colors[i], edgecolor=colors[i], alpha=0.7)
        ax.text(-0.05, y_lower + 0.5 * size, f'C{cluster_id}', fontsize=10, fontweight='bold')
        y_lower = y_upper + 10

    ax.axvline(x=sil_avg, color='red', linestyle='--', linewidth=2,
               label=f'Promedio: {sil_avg:.3f}')
    ax.set_xlabel('Coeficiente de Silueta', fontsize=11)
    ax.set_ylabel('Muestras por Cluster', fontsize=11)
    ax.set_title(f'Silhouette Plot - {nombre_m}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("Interpretación del Silhouette Plot:")
print("  - Cada barra horizontal representa un punto del dataset.")
print("  - Puntos con silueta positiva alta están bien asignados a su cluster.")
print("  - Puntos con silueta negativa están más cerca del cluster vecino (mal asignados).")
print("  - La línea roja punteada indica el promedio global.")

### 4d. Análisis de estabilidad de clusters

¿Qué tan estables son los clusters? Si cambiamos la semilla aleatoria o hacemos bootstrap, ¿obtenemos las mismas asignaciones? Un clustering estable es uno donde los mismos puntos terminan en los mismos clusters independientemente de la inicialización.

Medimos la estabilidad usando el **ARI promedio entre pares de ejecuciones** con diferentes semillas.

In [ ]:
# --- Análisis de estabilidad ---
n_runs = 20
seeds = range(n_runs)

print("=" * 70)
print("ANÁLISIS DE ESTABILIDAD DE CLUSTERS")
print("=" * 70)

algoritmos_estabilidad = {
    'K-Means': lambda s, X: KMeans(n_clusters=2, random_state=s, n_init=10).fit_predict(X),
    'Mini-Batch KM': lambda s, X: MiniBatchKMeans(n_clusters=2, random_state=s, batch_size=100, n_init=10).fit_predict(X),
}

for nombre_est, func_est in algoritmos_estabilidad.items():
    for X_data, data_name in [(X_scaled, 'Sin PCA'), (X_pca, 'Con PCA')]:
        # Generar etiquetas con diferentes semillas
        all_labels = [func_est(s, X_data) for s in seeds]

        # Calcular ARI entre todos los pares
        aris_pares = []
        for i in range(n_runs):
            for j in range(i+1, n_runs):
                aris_pares.append(adjusted_rand_score(all_labels[i], all_labels[j]))

        ari_mean = np.mean(aris_pares)
        ari_std = np.std(aris_pares)

        if ari_mean > 0.95:
            estabilidad = "Muy estable"
        elif ari_mean > 0.80:
            estabilidad = "Estable"
        elif ari_mean > 0.50:
            estabilidad = "Moderadamente estable"
        else:
            estabilidad = "Inestable"

        print(f"  {nombre_est:20s} ({data_name}): ARI medio = {ari_mean:.4f} ± {ari_std:.4f} → {estabilidad}")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (X_data, data_name) in enumerate([(X_scaled, 'Sin PCA'), (X_pca, 'Con PCA')]):
    ax = axes[ax_idx]
    # K-Means con 4 semillas diferentes
    for s_idx, seed in enumerate([0, 7, 42, 99]):
        labels_s = KMeans(n_clusters=2, random_state=seed, n_init=10).fit_predict(X_data)
        ax.scatter(X_2d[:, 0], X_2d[:, 1], c=labels_s, cmap='Set1',
                   alpha=0.3, s=10, edgecolors='none')
    ax.set_title(f'K-Means 4 semillas superpuestas ({data_name})', fontsize=12, fontweight='bold')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
    ax.grid(True, alpha=0.3)

plt.suptitle('Estabilidad Visual: ¿Cambian los clusters con diferentes semillas?',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nSi las 4 ejecuciones producen las mismas asignaciones, los puntos se superponen")
print("perfectamente. Diferencias visibles indicarían inestabilidad.")

### 4d. Análisis: ¿El clustering diferencia correctamente la variable "Survived"?

Dado que conocemos la variable objetivo real (Survived) de las unidades anteriores, evaluamos si los clusters coinciden con la supervivencia.

In [ ]:
# --- Análisis de correspondencia Cluster vs Survived ---
print("=" * 70)
print("ANÁLISIS DE CORRESPONDENCIA: CLUSTERS vs SURVIVED")
print("=" * 70)

def analizar_correspondencia(labels, y_real, nombre):
    print(f"\n--- {nombre} ---")
    mask = labels != -1
    labels_clean = labels[mask]
    y_clean = y_real.values[mask]

    if len(set(labels_clean)) < 2:
        print("  Solo 1 cluster detectado, no se puede analizar.")
        return

    # Crosstab
    ct = pd.crosstab(labels_clean, y_clean, margins=True)
    ct.index = [f'Cluster {i}' if i != 'All' else 'Total' for i in ct.index]
    ct.columns = ['No Sobrevivió', 'Sobrevivió', 'Total']
    print(ct)

    print(f"\n  Tasa de supervivencia por cluster:")
    for cluster in sorted(set(labels_clean)):
        mask_c = labels_clean == cluster
        tasa = y_clean[mask_c].mean()
        n = mask_c.sum()
        print(f"    Cluster {cluster}: {tasa*100:.1f}% sobrevivieron ({n} pasajeros)")

    if len(set(labels_clean)) == 2:
        acc_direct = accuracy_score(y_clean, labels_clean)
        acc_invert = accuracy_score(y_clean, 1 - labels_clean)
        best_acc = max(acc_direct, acc_invert)
        mapping = "directa" if acc_direct >= acc_invert else "invertida"
        print(f"\n  Accuracy de asignación (mejor mapping {mapping}): {best_acc*100:.1f}%")
        print(f"  ARI: {adjusted_rand_score(y_clean, labels_clean):.4f}")
        print(f"  NMI: {normalized_mutual_info_score(y_clean, labels_clean):.4f}")

        if mapping == "invertida":
            labels_mapped = 1 - labels_clean
        else:
            labels_mapped = labels_clean

        cm = confusion_matrix(y_clean, labels_mapped)
        print(f"\n  Matriz de Confusión (mejor mapping):")
        print(f"               Pred: No  Pred: Sí")
        print(f"    Real: No    {cm[0,0]:5d}     {cm[0,1]:5d}")
        print(f"    Real: Sí    {cm[1,0]:5d}     {cm[1,1]:5d}")

analizar_correspondencia(labels_kmeans_sp, y_true, "K-Means (Sin PCA)")
analizar_correspondencia(labels_kmeans_cp, y_true, "K-Means (Con PCA)")
analizar_correspondencia(labels_mejor, y_true, f"{mejor} ({mejor_datos}) - MEJOR MODELO")

In [ ]:
# --- Visualización: Clusters vs Survived ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

scatter0 = axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=y_true, cmap='coolwarm',
                            alpha=0.5, s=20, edgecolors='none')
axes[0].set_title('Ground Truth (Survived)', fontsize=12, fontweight='bold')
axes[0].legend(*scatter0.legend_elements(), title='Survived')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].grid(True, alpha=0.3)

scatter1 = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=labels_kmeans_sp, cmap='Set1',
                            alpha=0.5, s=20, edgecolors='none')
axes[1].set_title('K-Means (Sin PCA)', fontsize=12, fontweight='bold')
axes[1].legend(*scatter1.legend_elements(), title='Cluster')
axes[1].set_xlabel('PC1')
axes[1].grid(True, alpha=0.3)

scatter2 = axes[2].scatter(X_2d[:, 0], X_2d[:, 1], c=labels_mejor, cmap='Set1',
                            alpha=0.5, s=20, edgecolors='none')
axes[2].set_title(f'{mejor} ({mejor_datos})', fontsize=12, fontweight='bold')
axes[2].legend(*scatter2.legend_elements(), title='Cluster')
axes[2].set_xlabel('PC1')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Comparación: Ground Truth vs Clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 4e. Perfilado de clusters: ¿Quién está en cada cluster?

El perfilado muestra las **características promedio** de cada cluster, permitiendo interpretar qué "tipo de pasajero" agrupa cada uno. Esto transforma resultados numéricos en insights accionables.

In [ ]:
# --- Perfilado del mejor modelo ---
if mejor_datos == 'Sin PCA':
    labels_perfil = resultados_sin_pca[mejor]['labels']
else:
    labels_perfil = resultados_con_pca[mejor]['labels']

# Crear DataFrame con datos originales (antes de escalar) y los labels
df_perfil = df_encoded.copy()
df_perfil['Cluster'] = labels_perfil
df_perfil['Survived'] = y_true.values

# Filtrar ruido
df_perfil_clean = df_perfil[df_perfil['Cluster'] != -1].copy()

# Estadísticas por cluster
print("=" * 70)
print(f"PERFILADO DE CLUSTERS - {mejor} ({mejor_datos})")
print("=" * 70)

perfil = df_perfil_clean.groupby('Cluster').agg({
    'Pclass': 'mean',
    'Age': 'mean',
    'SibSp': 'mean',
    'Parch': 'mean',
    'Fare': 'mean',
    'FamilySize': 'mean',
    'IsAlone': 'mean',
    'Sex_male': 'mean',
    'Survived': ['mean', 'count']
})

perfil.columns = ['Clase Promedio', 'Edad Promedio', 'SibSp Promedio', 'Parch Promedio',
                   'Tarifa Promedio', 'Familia Promedio', '% Viajan Solos',
                   '% Hombres', '% Supervivencia', 'N Pasajeros']

print(perfil.round(3).to_string())

# Interpretación narrativa
print("\n--- Interpretación ---")
for cluster_id in sorted(df_perfil_clean['Cluster'].unique()):
    row = perfil.loc[cluster_id]
    n = int(row['N Pasajeros'])
    pct_male = row['% Hombres'] * 100
    pct_surv = row['% Supervivencia'] * 100
    clase = row['Clase Promedio']
    edad = row['Edad Promedio']
    fare = row['Tarifa Promedio']

    genero_desc = "mayoritariamente hombres" if pct_male > 60 else (
        "mayoritariamente mujeres" if pct_male < 40 else "mixto en género")
    clase_desc = "primera/segunda clase" if clase < 2.3 else "tercera clase" if clase > 2.7 else "clase media"

    print(f"\n  Cluster {cluster_id} ({n} pasajeros):")
    print(f"    Perfil: {genero_desc}, {clase_desc}, edad promedio {edad:.0f} años, tarifa ${fare:.1f}")
    print(f"    Supervivencia real: {pct_surv:.1f}%")

In [ ]:
# --- Gráfico Radar para perfilado visual ---
# Normalizar las métricas del perfil para el radar
categorias_radar = ['Clase', 'Edad', 'Tarifa', '% Hombres', 'Familia', '% Solos']

# Obtener valores por cluster (normalizados 0-1 para el radar)
clusters_ids = sorted(df_perfil_clean['Cluster'].unique())
n_clusters_perfil = len(clusters_ids)

valores_radar = {}
for c_id in clusters_ids:
    mask_c = df_perfil_clean['Cluster'] == c_id
    vals = [
        df_perfil_clean.loc[mask_c, 'Pclass'].mean() / 3,
        df_perfil_clean.loc[mask_c, 'Age'].mean() / df_perfil_clean['Age'].max(),
        df_perfil_clean.loc[mask_c, 'Fare'].mean() / df_perfil_clean['Fare'].max(),
        df_perfil_clean.loc[mask_c, 'Sex_male'].mean(),
        df_perfil_clean.loc[mask_c, 'FamilySize'].mean() / df_perfil_clean['FamilySize'].max(),
        df_perfil_clean.loc[mask_c, 'IsAlone'].mean(),
    ]
    valores_radar[c_id] = vals

# Crear radar chart
angles = np.linspace(0, 2 * np.pi, len(categorias_radar), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors_radar = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

for i, c_id in enumerate(clusters_ids):
    valores = valores_radar[c_id] + valores_radar[c_id][:1]
    ax.plot(angles, valores, 'o-', linewidth=2, label=f'Cluster {c_id}',
            color=colors_radar[i % len(colors_radar)])
    ax.fill(angles, valores, alpha=0.1, color=colors_radar[i % len(colors_radar)])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categorias_radar, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title(f'Perfil de Clusters - {mejor} ({mejor_datos})', fontsize=14,
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("El gráfico radar permite comparar visualmente los perfiles de cada cluster.")
print("Valores más altos en '% Hombres' y 'Clase' (cercano a 3=tercera clase)")
print("tienden a asociarse con menor supervivencia.")

### 4g. Distribuciones por cluster (Violin Plots)

El perfilado con promedios puede ocultar la dispersión interna de cada cluster. Los **violin plots** muestran la distribución completa de cada variable por cluster, revelando bimodalidades, asimetrías y outliers que los promedios no capturan.

In [ ]:
# --- Violin plots de variables clave por cluster ---
variables_dist = ['Pclass', 'Age', 'Fare', 'FamilySize']
n_vars = len(variables_dist)

fig, axes = plt.subplots(2, n_vars, figsize=(20, 10))

# Fila 1: Por cluster
for idx, var in enumerate(variables_dist):
    ax = axes[0, idx]
    data_violin = []
    labels_violin = []
    for c_id in sorted(df_perfil_clean['Cluster'].unique()):
        vals = df_perfil_clean.loc[df_perfil_clean['Cluster'] == c_id, var].values
        data_violin.append(vals)
        labels_violin.append(f'Cluster {c_id}')

    parts = ax.violinplot(data_violin, showmeans=True, showmedians=True)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(['#3498db', '#e74c3c', '#2ecc71', '#f39c12'][i % 4])
        pc.set_alpha(0.6)
    ax.set_xticks(range(1, len(labels_violin)+1))
    ax.set_xticklabels(labels_violin, fontsize=9)
    ax.set_title(f'{var} por Cluster', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

# Fila 2: Por Survived (para comparar)
for idx, var in enumerate(variables_dist):
    ax = axes[1, idx]
    data_violin = []
    labels_violin = ['No Sobrevivió', 'Sobrevivió']
    for surv in [0, 1]:
        vals = df_perfil_clean.loc[df_perfil_clean['Survived'] == surv, var].values
        data_violin.append(vals)

    parts = ax.violinplot(data_violin, showmeans=True, showmedians=True)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(['#95a5a6', '#27ae60'][i])
        pc.set_alpha(0.6)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(labels_violin, fontsize=9)
    ax.set_title(f'{var} por Survived', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Distribuciones: Clusters ({mejor}) vs Survived (Ground Truth)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("Interpretación:")
print("  Fila superior: distribución de cada variable dentro de cada cluster.")
print("  Fila inferior: distribución de cada variable según supervivencia real.")
print("  Si los patrones son similares entre filas, el clustering está capturando")
print("  la misma estructura que define la supervivencia.")

In [ ]:
# --- Heatmap resumen final ---
fig, ax = plt.subplots(figsize=(14, 5))

metricas_hm = ['Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin', 'ARI', 'NMI']
data_hm = {}

for nombre in nombres_algo:
    for datos, df_m in [('Sin PCA', df_metricas_sp), ('Con PCA', df_metricas_cp)]:
        key = f"{nombre}\n({datos})"
        vals = [df_m.loc[nombre, m] if nombre in df_m.index else np.nan for m in metricas_hm]
        data_hm[key] = vals

df_hm = pd.DataFrame(data_hm, index=metricas_hm)

sns.heatmap(df_hm, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=1,
            linecolor='white', ax=ax, center=0.3)
ax.set_title('Mapa de Calor: Todas las Métricas por Modelo y Datos', fontsize=14, fontweight='bold')
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

---

## 5. Repositorio GitHub

Para cargar el cuaderno Python en GitHub, se ejecutan los siguientes comandos desde la terminal:

```bash
# Inicializar repositorio (si no existe)
git init

# Agregar archivos
git add Titanic_ML_Unidad3.ipynb

# Crear commit
git commit -m "Add Trabajo Sumativo N°3 - Clustering y Aprendizaje No Supervisado"

# Agregar remote (reemplazar con URL real del equipo)
git remote add origin https://github.com/usuario/titanic-ml-umayor.git

# Push
git push -u origin main
```

El cuaderno queda versionado y accesible para todos los integrantes del equipo.

---

## Conclusión

En este tercer y último trabajo aplicamos técnicas de **aprendizaje no supervisado (clustering)** al dataset del Titanic, abordando el problema desde una perspectiva completamente diferente a las unidades anteriores.

**Sobre la preparación de datos:**
- Mantuvimos consistencia con el pipeline de las unidades previas (limpieza, imputación, winsorización).
- Aplicamos One Hot Encoding para convertir variables categóricas en numéricas, necesario para los algoritmos basados en distancia.
- El análisis de correlación reveló redundancia entre SibSp, Parch, FamilySize e IsAlone, lo que justifica el uso de PCA para comprimir esa información.
- La normalización con StandardScaler fue esencial para igualar la contribución de cada variable al cálculo de distancias.

**Sobre PCA:**
- La reducción de dimensionalidad permitió capturar la mayor parte de la varianza en menos componentes.
- El heatmap de loadings reveló que cada componente captura un "concepto" diferente del pasajero (perfil socioeconómico, composición familiar, puerto de embarque, etc.).
- La comparación de umbrales de varianza mostró cómo la calidad del clustering varía con el número de componentes, permitiendo una selección informada.

**Sobre los algoritmos de clustering:**
- Los algoritmos basados en distancia (K-Means, Mini-Batch K-Means, Agglomerative) generaron particiones más estables y predecibles.
- Los algoritmos basados en densidad (DBSCAN, OPTICS, Mean Shift) fueron útiles para detectar outliers y estructuras de forma irregular, pero requirieron ajuste cuidadoso de hiperparámetros. El gráfico de k-distancias y el tuning sistemático de DBSCAN permitieron una selección fundamentada de eps y min_samples.
- El análisis de tiempos de ejecución demostró que Mini-Batch K-Means es el más eficiente, mientras que OPTICS y Mean Shift son los más costosos computacionalmente.

**Sobre la evaluación y perfilado:**
- El silhouette plot por muestra reveló la calidad individual de cada asignación, más allá del promedio.
- El perfilado de clusters y el gráfico radar transformaron los resultados numéricos en insights interpretables sobre qué "tipo de pasajero" caracteriza cada cluster.
- Los clusters encontrados de forma no supervisada logran capturar parcialmente la estructura que la variable Survived define, pero la correspondencia no es perfecta, lo que refuerza la idea de que el aprendizaje supervisado siempre tendrá ventaja cuando se dispone de datos etiquetados.

**Aprendizaje clave:** El clustering es una herramienta poderosa para descubrir estructura oculta en los datos, especialmente cuando no se dispone de etiquetas. El perfilado de los clusters y la comparación con las etiquetas reales nos permite validar si los patrones descubiertos tienen significado real, y la combinación de métricas internas y externas ofrece una evaluación robusta de la calidad del agrupamiento.

---

## Bibliografía

- MacQueen, J. (1967). *Some methods for classification and analysis of multivariate observations*. Proceedings of the 5th Berkeley Symposium on Mathematical Statistics and Probability, 1, 281-297.

- Ester, M., Kriegel, H.P., Sander, J. & Xu, X. (1996). *A density-based algorithm for discovering clusters in large spatial databases with noise*. Proceedings of the 2nd International Conference on Knowledge Discovery and Data Mining (KDD-96), 226-231.

- Ankerst, M., Breunig, M.M., Kriegel, H.P. & Sander, J. (1999). *OPTICS: Ordering Points To Identify the Clustering Structure*. ACM SIGMOD Record, 28(2), 49-60.

- Comaniciu, D. & Meer, P. (2002). *Mean Shift: A Robust Approach Toward Feature Space Analysis*. IEEE Transactions on Pattern Analysis and Machine Intelligence, 24(5), 603-619.

- Ward, J.H. (1963). *Hierarchical Grouping to Optimize an Objective Function*. Journal of the American Statistical Association, 58(301), 236-244.

- Jolliffe, I.T. (2002). *Principal Component Analysis* (2nd ed.). Springer. https://doi.org/10.1007/b98835

- Rousseeuw, P.J. (1987). *Silhouettes: A graphical aid to the interpretation and validation of cluster analysis*. Journal of Computational and Applied Mathematics, 20, 53-65.

- Caliński, T. & Harabasz, J. (1974). *A dendrite method for cluster analysis*. Communications in Statistics, 3(1), 1-27.

- Davies, D.L. & Bouldin, D.W. (1979). *A Cluster Separation Measure*. IEEE Transactions on Pattern Analysis and Machine Intelligence, 1(2), 224-227.

- Scikit-learn developers. (s.f.). *Scikit-learn: Machine Learning in Python*. https://scikit-learn.org/stable/

- Kaggle. (s.f.). *Titanic - Machine Learning from Disaster*. https://www.kaggle.com/c/titanic